In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
import pickle
from sklearn.multiclass import OneVsRestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'
new_clusters = pd.read_csv(f'{indir}/csv/label_transfer/snmct_xgboost_v6.csv', sep='\t', header=0, index_col=0)
new_clusters.index = [f'{x}_indexed' for x in new_clusters.index]
new_clusters.shape

In [ ]:
new_clusters.head()

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/chrom100k/merged.processed.h5ad')

In [ ]:
adata

In [ ]:
adata.obs['cluster'] = new_clusters.reindex(adata.obs.index)['pred_coarse']

In [ ]:
hexes = ['#1D8348', '#5B2C6F', '#922B21', '#F1C40F']
adata.uns['donor_colors'] = hexes

In [ ]:
adata.obs['new_cluster'] = new_clusters.reindex(adata.obs.index)['pred_coarse']
new_adata = adata[~adata.obs['new_cluster'].isna()]
sc.pl.umap(new_adata, color=['donor', 'time_num', 'cluster'], ncols=2)

### load all of the feature sets in

In [ ]:
n_hvf = 20000
train_size = 0.8
test_size = 0.2
model = OneVsRestClassifier(XGBClassifier(eval_metric='logloss', random_state=42))

In [ ]:
chrom5k_adata = sc.read_h5ad(f'{projdir}/h5ad/bins/merged.chrom5k.processed.h5ad')
chrom5k_adata.obs.index = [f'{x}_indexed' for x in chrom5k_adata.obs.index]

In [ ]:
chrom5k_adata

In [ ]:
enh_adata = sc.read_h5ad(f'{projdir}/h5ad/peaks/enhancers_norm_processed.h5ad')

In [ ]:
enh_adata

In [ ]:
genes_adata = sc.read_h5ad(f'{projdir}/h5ad/genebody/merged.processed.h5ad')

In [ ]:
genes_adata

In [ ]:
dmrs_adata = sc.read_h5ad(f'{projdir}/h5ad/dmrs/rna_cluster_dmrs.processed.h5ad')
dmrs_adata.obs.index = [f'{x}_indexed' for x in dmrs_adata.obs.index]

In [ ]:
dmrs_adata

In [ ]:
peaks_adata = sc.read_h5ad(f'{projdir}/h5ad/peaks/peaks_hvf_processed.h5ad')
peaks_adata.obs.index = [f'{x}_indexed' for x in peaks_adata.obs.index]

In [ ]:
peaks_adata

### subset to clusters and rerun pca

In [ ]:
mask = chrom5k_adata.obs.index.isin(new_clusters.index)
final_chrom5k_adata = chrom5k_adata[mask]
final_chrom5k_adata.obs['cluster'] = new_clusters.reindex(final_chrom5k_adata.obs.index)['pred_coarse']
sc.pp.highly_variable_genes(final_chrom5k_adata, n_top_genes=n_hvf)
sc.tl.pca(final_chrom5k_adata, n_comps=50)

In [ ]:
mask = enh_adata.obs.index.isin(new_clusters.index)
final_enh_adata = enh_adata[mask]
final_enh_adata.obs['cluster'] = new_clusters.reindex(final_enh_adata.obs.index)['pred_coarse']
sc.pp.highly_variable_genes(final_enh_adata, n_top_genes=n_hvf)
sc.tl.pca(final_enh_adata, n_comps=50)

In [ ]:
mask = genes_adata.obs.index.isin(new_clusters.index)
final_genes_adata = genes_adata[mask]
final_genes_adata.obs['cluster'] = new_clusters.reindex(final_genes_adata.obs.index)['pred_coarse']
sc.pp.highly_variable_genes(final_genes_adata, n_top_genes=n_hvf)
sc.tl.pca(final_genes_adata, n_comps=50)

In [ ]:
mask = peaks_adata.obs.index.isin(new_clusters.index)
final_peaks_adata = peaks_adata[mask]
final_peaks_adata.obs['cluster'] = new_clusters.reindex(final_peaks_adata.obs.index)['pred_coarse']
sc.pp.highly_variable_genes(final_peaks_adata, n_top_genes=n_hvf)
sc.tl.pca(final_peaks_adata, n_comps=50)

In [ ]:
mask = dmrs_adata.obs.index.isin(new_clusters.index)
final_dmrs_adata = dmrs_adata[mask]
final_dmrs_adata.obs['cluster'] = new_clusters.reindex(final_dmrs_adata.obs.index)['pred_coarse']
sc.pp.highly_variable_genes(final_dmrs_adata, n_top_genes=n_hvf)
sc.tl.pca(final_dmrs_adata, n_comps=50)

### write everything out

In [ ]:
final_genes_adata.write_h5ad(f'{projdir}/h5ad/pred_rna/genes_adata.h5ad')

In [ ]:
final_enh_adata.write_h5ad(f'{projdir}/h5ad/pred_rna/enh_adata.h5ad')

In [ ]:
final_peaks_adata.write_h5ad(f'{projdir}/h5ad/pred_rna/peaks_adata.h5ad')

In [ ]:
final_chrom5k_adata.write_h5ad(f'{projdir}/h5ad/pred_rna/chrom5k_adata.h5ad')

In [ ]:
final_dmrs_adata.write_h5ad(f'{projdir}/h5ad/pred_rna/dmrs_adata.h5ad')

In [ ]:
!date